# Synthetic Input Builder + Geoprivacy Demo (public - no real data)

Every cell here **runs in public** and reads **no real data at any point**.

The pipeline normally reads its input from `data/`. On a real machine that input is the confidential 
`data/input_data.xlsx`. Here we instead place a **synthetic** file, `data/synthetic_input_data.xlsx`, 
in that same input folder and treat it as if it were the real local data. This notebook then does two 
things with it: it generates a **second** synthetic file, `data/synthetic_input_data_double.xlsx`, which 
the pipeline actually tests on, and it demonstrates the geographic-privacy tools from the Map Encryption 
Library (Jim & Herman, `PHI-Case-Studies/2026-Map-Encryption-Library`) on the input coordinates. When it 
finishes, you run the pipeline on the generated file.

### Why nothing here is confidential

The real, confidential data is used **only once, offline**, by a separate local notebook, to produce the 
synthetic input `data/synthetic_input_data.xlsx`. That file already contains no real record: its 
coordinates are random points inside the correct oblast (province) and its case counts are binomial draws 
from non-personal aggregates. This notebook then produces a further synthetic file from it, one more step 
removed from the real data. No individual real record and no exact real location can be reconstructed from 
either file; what they deliberately preserve is coarse aggregate shape (the recent-infection share per 
anonymised site per month), which is anonymised to province level and already appears in the published 
results.

## Step 1 - Load the input and reduce it to non-personal aggregates  *(runs-in-public)*

We read the synthetic input from the `data/` folder (it stands in for the real local data) and reuse the 
project's own extractor to reduce it to per-site, per-month counts.

In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, os.getcwd())   # notebook lives in synthetic_data/
import pandas as pd

INPUT = Path('..') / 'data' / 'synthetic_input_data.xlsx'   # synthetic stand-in for the real local data
assert INPUT.exists(), 'place data/synthetic_input_data.xlsx in the input folder first'

import extract_site_profiles as extract
extract.EXCEL       = INPUT                          # <-- read the synthetic input, never a real file
extract.OUT         = Path('agg_site_profiles.csv')
extract.OUT_MONTHLY = Path('agg_site_monthly.csv')
extract.main()
print('\nAggregates computed from the synthetic input (no real data touched).')

## Step 2 - Generate the synthetic file the pipeline will test on  *(runs-in-public)*

The generator invents a fresh random coordinate inside each site's province and re-draws the monthly case 
counts, then writes the result straight into the `data/` input folder. This is the file the pipeline runs 
on. We never overwrite anything called `input_data.xlsx`.

In [ ]:
import generate_synthetic as generate
generate.PROFILES = Path('agg_site_profiles.csv')
generate.MONTHLY  = Path('agg_site_monthly.csv')
TEST_INPUT = Path('..') / 'data' / 'synthetic_input_data_double.xlsx'
assert TEST_INPUT.name != 'input_data.xlsx', 'refusing to overwrite the real input filename'
generate.OUT = TEST_INPUT
generate.main()

inp  = pd.read_excel(INPUT, sheet_name='hiv_cases')
test = pd.read_excel(TEST_INPUT, sheet_name='hiv_cases')
print('\nrecent share - input: %.4f | generated test file: %.4f'
      % ((inp['type']=='recent').mean(), (test['type']=='recent').mean()))
print('pipeline test input ->', TEST_INPUT.resolve())
print('To run the pipeline: set config.json  excel_path -> "data/synthetic_input_data_double.xlsx"')

## Step 3 - Geoprivacy demo on the input coordinates  *(runs-in-public)*

We now treat the input file as if it were real data that needs protecting, and demonstrate the two tools 
from the Map Encryption Library. Clone `PHI-Case-Studies/2026-Map-Encryption-Library` next to this 
repository (or install it) so the import can find it; the generation steps above do not need it.

### 3a. Encryption: encode -> decode (with key) -> scatter (without key)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
try:
    try:
        from map_encryption import MapEncryption, SchemeParams
    except ModuleNotFoundError:
        for cand in ['../2026-Map-Encryption-Library', './2026-Map-Encryption-Library',
                     '../../2026-Map-Encryption-Library', '../../../2026-Map-Encryption-Library']:
            if os.path.exists(os.path.join(cand, 'map_encryption', '__init__.py')):
                sys.path.insert(0, cand); break
        from map_encryption import MapEncryption, SchemeParams
    _HAVE_ENC = True
except ModuleNotFoundError:
    _HAVE_ENC = False
    print('Map Encryption Library not found. Clone PHI-Case-Studies/2026-Map-Encryption-Library next to '
          'this repo (or install it) to run the encryption demo. Steps 1-2 above do not need it.')

if _HAVE_ENC:
  sample = inp.sample(n=min(800, len(inp)), random_state=2026).reset_index(drop=True)
  lat = sample['latitude'].to_numpy(); lon = sample['longitude'].to_numpy()
  key = bytes.fromhex('00112233445566778899aabbccddeeff' * 2)   # DEMO key; real use = secret random key
  enc = MapEncryption(key, SchemeParams(tile_system='mercator', bin_size_m=250))
  records = [enc.encode(float(lat[i]), float(lon[i]),
                        tweak=MapEncryption.make_tweak(record_id=i, extra=b'input-demo'))
             for i in range(len(sample))]
  decoded = [enc.decode(r) for r in records]
  max_err = max(max(abs(decoded[i][0]-lat[i]), abs(decoded[i][1]-lon[i])) for i in range(len(sample)))
  disp = np.array([enc.render_coordinates(r) for r in records])
  print(f'round-trip max error (with key): {max_err:.2e} deg -> exact recovery of the coordinate')
  print('display without key -> scattered across the globe: lat[%.0f,%.0f] lon[%.0f,%.0f]'
        % (disp[:,0].min(), disp[:,0].max(), disp[:,1].min(), disp[:,1].max()))
  fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
  ax[0].scatter(lon, lat, s=6, c='#17345E'); ax[0].set_title('Input coordinates (Ukraine)')
  ax[1].scatter(disp[:,1], disp[:,0], s=6, c='#FB8500'); ax[1].set_title('Encrypted display (no key)')
  ax[1].set_xlim(-180,180); ax[1].set_ylim(-90,90)
  for a in ax: a.set_xlabel('lon'); a.set_ylabel('lat')
  plt.tight_layout(); plt.show()

### 3b. Donut geomasking (optional) - a structure-*preserving* alternative

Donut geomasking nudges each point a short random distance, so population-level structure survives. 
Requires `geopy`; the cell degrades gracefully if it is absent.

In [ ]:
try:
    import numpy as np
    from geoprivacy.donut_geomask import donut_geomask
    ds = inp.sample(n=50, random_state=7).reset_index(drop=True)
    band = [(50, 100), (100, 300)]
    dist_m = np.array([donut_geomask(band, (float(ds['latitude'][i]), float(ds['longitude'][i])))['distance'] * 1000
                       for i in range(len(ds))])
    print('donut geomasking on 50 points: mean %.0f m, max %.0f m -> points stay nearby, pattern preserved'
          % (dist_m.mean(), dist_m.max()))
except ModuleNotFoundError:
    print('geopy not installed - skipping donut demo. Install geopy to run it.')

## Summary and an honest privacy statement

The pipeline can now be run on `data/synthetic_input_data_double.xlsx`, produced above from a synthetic 
input, with no real data involved at any step.

**Stated honestly, not as an absolute guarantee:** no individual real record and no exact real location 
can be reconstructed from these synthetic files, because coordinates are invented random points within a 
province and dates are drawn within a month. What is deliberately preserved is coarse aggregate shape - the 
approximate recent-infection share per anonymised site per month - which is anonymised to province level and 
already appears in the published results. Under any realistic threat model the synthetic files expose no real 
individual and no real place. Building-snapping was intentionally omitted so coordinates stay unrelated to any 
real location.